# Pipeline 3: Reintegration Readiness Prediction

## New Dawn Lighthouse Foundation - Predictive Classification Model

**Business Problem:** Reintegration -- the process of transitioning a trafficking survivor from safehouse care back to family or community life -- is the most consequential decision in the Lighthouse Foundation's case management workflow. Premature reintegration can expose a child to re-trafficking, abuse, or neglect. Delayed reintegration keeps a child in institutional care longer than necessary, potentially impeding their social development and occupying safehouse capacity needed by other survivors.

This pipeline builds a **predictive classification model** to assess reintegration readiness based on multi-dimensional progress indicators aggregated from process recordings, education records, health assessments, intervention plans, home visitations, and incident reports. The model serves as a **decision-support tool** for social workers -- it does not replace professional judgment but provides a data-driven second opinion.

**Target Variable:** `reintegration_completed` - binary indicator (1 = reintegration_status is 'Completed', 0 = otherwise).

**Critical Constraint:** **Precision is the priority metric.** A false positive (predicting a resident is ready when they are not) could result in premature reintegration with severe consequences. We would rather miss some ready residents (lower recall) than incorrectly clear an unready resident (lower precision).

**Approach:**
- **Ch. 6-7:** Multi-table feature engineering and profiling.
- **Ch. 8:** Bivariate analysis of aggregated features vs. target.
- **Ch. 9:** Statsmodels logistic regression for explanatory insight.
- **Ch. 10-12:** sklearn classifiers with StratifiedKFold, optimizing for precision.
- **Ch. 13-14:** GridSearchCV and feature importance.
- **Ch. 15:** Deployment to `models/reintegration_model.pkl`.

---
## Section 1: Data Loading & Multi-Table Feature Engineering (Ch. 5, 7)

This pipeline involves the most complex feature engineering of all four models because the predictive signal is distributed across **seven tables**. Each table captures a different dimension of a resident's progress:

| Table | Dimension | Key Metrics |
|-------|-----------|-------------|
| residents | Demographics & case info | length_of_stay, risk_level, case_category |
| process_recordings | Counseling progress | session count, emotional improvement |
| education_records | Academic progress | progress_percent, attendance_rate |
| health_wellbeing_records | Physical/mental health | health scores, checkup completion |
| intervention_plans | Treatment completion | plan status, completion rate |
| home_visitations | Family assessment | cooperation level, safety concerns |
| incident_reports | Behavioral issues | incident count, severity |

We aggregate all tables to the **resident level** (one row per resident) and join on `resident_id`.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

import sys
sys.path.insert(0, '.')
from functions import unistats, bin_categories, skew_correct, missing_fill, clean_outlier, n2n_analysis, n2c_analysis, c2c_analysis, calculate_vif

print('Libraries loaded successfully.')

Libraries loaded successfully.


In [2]:
# Load all relevant tables
residents = pd.read_csv('../lighthouse_csv_v7/residents.csv')
process_recs = pd.read_csv('../lighthouse_csv_v7/process_recordings.csv', parse_dates=['session_date'])
education = pd.read_csv('../lighthouse_csv_v7/education_records.csv', parse_dates=['record_date'])
health = pd.read_csv('../lighthouse_csv_v7/health_wellbeing_records.csv', parse_dates=['record_date'])
interventions = pd.read_csv('../lighthouse_csv_v7/intervention_plans.csv')
visitations = pd.read_csv('../lighthouse_csv_v7/home_visitations.csv', parse_dates=['visit_date'])
incidents = pd.read_csv('../lighthouse_csv_v7/incident_reports.csv', parse_dates=['incident_date'])

print(f'Residents:          {residents.shape}')
print(f'Process Recordings: {process_recs.shape}')
print(f'Education Records:  {education.shape}')
print(f'Health Records:     {health.shape}')
print(f'Intervention Plans: {interventions.shape}')
print(f'Home Visitations:   {visitations.shape}')
print(f'Incident Reports:   {incidents.shape}')

Residents:          (60, 49)
Process Recordings: (2819, 15)
Education Records:  (534, 10)
Health Records:     (534, 14)
Intervention Plans: (180, 11)
Home Visitations:   (1337, 14)
Incident Reports:   (100, 12)


In [3]:
# Examine target variable distribution
print('Reintegration Status Distribution:')
print(residents['reintegration_status'].value_counts())
print(f'\nTotal residents: {len(residents)}')

Reintegration Status Distribution:
reintegration_status
In Progress    21
Completed      19
On Hold        13
Not Started     7
Name: count, dtype: int64

Total residents: 60


In [4]:
residents.head(3)

,resident_id,case_control_no,internal_code,safehouse_id,case_status,sex,date_of_birth,birth_status,place_of_birth,religion,...,initial_case_assessment,date_case_study_prepared,reintegration_type,reintegration_status,initial_risk_level,current_risk_level,date_enrolled,date_closed,created_at,notes_restricted
0,1,C0043,LS-0001,4,Active,F,2008-08-31,Marital,Davao City,Unspecified,...,For Reunification,2023-12-14,Foster Care,In Progress,Critical,High,2023-10-17,NaN,2023-10-17 00:00:00,NaN
1,2,C2530,LS-0002,3,Closed,F,2008-04-23,Marital,Cebu City,Seventh-day Adventist,...,For Continued Care,2023-04-10,Family Reunification,Completed,Medium,Medium,2023-03-18,2025-01-06,2023-03-18 00:00:00,NaN
2,3,C3946,LS-0003,1,Active,F,2007-01-31,Marital,Manila,Roman Catholic,...,For Independent Living,NaN,Foster Care,Completed,Medium,Medium,2024-05-24,NaN,2024-05-24 00:00:00,NaN


### 1.1 Feature Engineering from Process Recordings

Process recordings capture counseling sessions between social workers and residents. We engineer features that measure both the *quantity* of counseling received and the *quality* of emotional progress:

- **total_counseling_sessions:** Total number of recorded sessions. More sessions indicate longer engagement with therapeutic support.
- **emotional_improvement_ratio:** Proportion of sessions where the resident's emotional state improved from start to end. This captures therapeutic responsiveness -- a resident who consistently shows emotional improvement in sessions is likely progressing.

In [5]:
# Process Recordings: counseling session features
print('Emotional state columns:')
print(f'  emotional_state_observed (start): {process_recs["emotional_state_observed"].unique()}')
print(f'  emotional_state_end: {process_recs["emotional_state_end"].unique()}')

# Define emotional state ordering for improvement detection
emotion_order = {'Distressed': 0, 'Anxious': 1, 'Withdrawn': 2, 'Guarded': 3, 
                 'Neutral': 4, 'Cooperative': 5, 'Calm': 6, 'Engaged': 7, 
                 'Hopeful': 8, 'Joyful': 9}

process_recs['emotion_start_score'] = process_recs['emotional_state_observed'].map(emotion_order)
process_recs['emotion_end_score'] = process_recs['emotional_state_end'].map(emotion_order)
process_recs['emotion_improved'] = (process_recs['emotion_end_score'] > process_recs['emotion_start_score']).astype(int)

counseling_features = process_recs.groupby('resident_id').agg(
    total_counseling_sessions=('recording_id', 'count'),
    mean_session_duration=('session_duration_minutes', 'mean'),
    emotional_improvement_ratio=('emotion_improved', 'mean')
).reset_index()

print(f'\nCounseling features: {counseling_features.shape}')
counseling_features.head()

Emotional state columns:
  emotional_state_observed (start): <StringArray>
[     'Angry', 'Distressed',    'Anxious',    'Hopeful',      'Happy',
       'Calm',        'Sad',  'Withdrawn']
Length: 8, dtype: str
  emotional_state_end: <StringArray>
['Hopeful', 'Sad', 'Happy', 'Calm', 'Anxious', 'Withdrawn']
Length: 6, dtype: str

Counseling features: (60, 4)


,resident_id,total_counseling_sessions,mean_session_duration,emotional_improvement_ratio
0,1,106,69.433962,0.358491
1,2,51,68.176471,0.372549
2,3,53,69.452830,0.377358
3,4,57,69.596491,0.438596
4,5,18,65.611111,0.333333


### 1.2 Feature Engineering from Education Records

Education records track academic progress over time. We engineer:
- **mean_education_progress:** Average progress_percent across all records. Higher values indicate consistent academic engagement.
- **education_trend_slope:** Linear regression slope of progress_percent over time. A positive slope means the resident's academic performance is improving, which is a strong readiness signal.
- **mean_attendance_rate:** Average attendance rate. Regular attendance reflects stability and routine.

In [6]:
# Education: progress features with trend
def calc_trend_slope(group, date_col, value_col):
    """Calculate linear trend slope for a time-series group."""
    if len(group) < 2:
        return 0.0
    group = group.sort_values(date_col).dropna(subset=[value_col])
    if len(group) < 2:
        return 0.0
    x = np.arange(len(group))
    y = group[value_col].values
    slope, _, _, _, _ = stats.linregress(x, y)
    return slope

edu_agg = education.groupby('resident_id').agg(
    mean_education_progress=('progress_percent', 'mean'),
    mean_attendance_rate=('attendance_rate', 'mean'),
    education_record_count=('education_record_id', 'count')
).reset_index()

# Calculate education trend slope per resident
edu_trends = education.groupby('resident_id').apply(
    lambda g: calc_trend_slope(g, 'record_date', 'progress_percent')
).reset_index(name='education_trend_slope')

edu_features = edu_agg.merge(edu_trends, on='resident_id', how='left')
print(f'Education features: {edu_features.shape}')
edu_features.head()

Education features: (60, 5)


,resident_id,mean_education_progress,mean_attendance_rate,education_record_count,education_trend_slope
0,1,45.483333,0.716333,6,3.037143
1,2,85.230000,0.834300,10,4.072121
2,3,71.581818,0.738091,11,6.059091
3,4,95.045455,0.757636,11,1.923636
4,5,61.388889,0.668111,9,4.965000


### 1.3 Feature Engineering from Health & Wellbeing Records

Health records track physical and mental wellbeing scores over time. We create a composite health score and measure improvement trends:
- **mean_health_score:** Average of general_health_score across all records.
- **health_trend_slope:** Linear trend of general_health_score over time. Positive slope indicates improving health.

In [7]:
# Health: composite score and trend
health_agg = health.groupby('resident_id').agg(
    mean_health_score=('general_health_score', 'mean'),
    mean_nutrition_score=('nutrition_score', 'mean'),
    mean_sleep_quality=('sleep_quality_score', 'mean'),
    health_record_count=('health_record_id', 'count')
).reset_index()

# Health trend slope
health_trends = health.groupby('resident_id').apply(
    lambda g: calc_trend_slope(g, 'record_date', 'general_health_score')
).reset_index(name='health_trend_slope')

health_features = health_agg.merge(health_trends, on='resident_id', how='left')
print(f'Health features: {health_features.shape}')
health_features.head()

Health features: (60, 6)


,resident_id,mean_health_score,mean_nutrition_score,mean_sleep_quality,health_record_count,health_trend_slope
0,1,3.103333,3.210000,3.203333,6,0.026286
1,2,3.449000,3.431000,3.376000,10,0.043576
2,3,3.181818,3.003636,3.079091,11,0.056455
3,4,3.157273,2.983636,2.881818,11,0.010818
4,5,3.087778,3.100000,2.981111,9,0.006167


### 1.4 Feature Engineering from Intervention Plans

Intervention plans document specific treatment goals and their completion status. The **intervention_completion_rate** -- the proportion of plans with status 'Completed' -- is a direct measure of treatment progress.

In [8]:
# Intervention Plans: completion rate
interventions['is_completed'] = (interventions['status'] == 'Completed').astype(int)

intervention_features = interventions.groupby('resident_id').agg(
    total_interventions=('plan_id', 'count'),
    intervention_completion_rate=('is_completed', 'mean'),
    intervention_categories=('plan_category', 'nunique')
).reset_index()

print(f'Intervention features: {intervention_features.shape}')
intervention_features.head()

Intervention features: (60, 4)


,resident_id,total_interventions,intervention_completion_rate,intervention_categories
0,1,3,0.0,3
1,2,3,0.0,3
2,3,3,0.0,3
3,4,3,0.0,3
4,5,3,0.0,3


### 1.5 Feature Engineering from Home Visitations

Home visitations assess the family environment's readiness to receive the resident. We extract:
- **home_visit_count:** Total visits conducted. More visits indicate thorough family assessment.
- **home_visit_cooperation_mode:** Most common family cooperation level. High cooperation suggests a supportive home environment.

In [9]:
# Home Visitations: family assessment features
visit_features = visitations.groupby('resident_id').agg(
    home_visit_count=('visitation_id', 'count'),
    safety_concerns_count=('safety_concerns_noted', lambda x: x.astype(str).str.lower().isin(['yes', 'true', '1']).sum()),
    follow_up_needed_count=('follow_up_needed', lambda x: x.astype(str).str.lower().isin(['yes', 'true', '1']).sum())
).reset_index()

# Family cooperation mode
coop_mode = visitations.groupby('resident_id')['family_cooperation_level'].agg(
    lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else 'Unknown'
).reset_index(name='home_visit_cooperation_mode')

visit_features = visit_features.merge(coop_mode, on='resident_id', how='left')
print(f'Visitation features: {visit_features.shape}')
visit_features.head()

Visitation features: (58, 5)


,resident_id,home_visit_count,safety_concerns_count,follow_up_needed_count,home_visit_cooperation_mode
0,1,54,9,27,Cooperative
1,2,35,11,17,Cooperative
2,3,26,11,12,Cooperative
3,4,9,3,3,Highly Cooperative
4,5,11,2,3,Highly Cooperative


### 1.6 Feature Engineering from Incident Reports

Incident reports document behavioral issues, safety events, and rule violations. We create:
- **incident_count:** Total number of incidents. Fewer incidents suggest better behavioral adjustment.
- **incident_severity_score:** Weighted severity score (e.g., High=3, Medium=2, Low=1). Higher scores indicate more serious behavioral concerns.

In [10]:
# Incident Reports: behavioral features
severity_map = {'Low': 1, 'Medium': 2, 'High': 3, 'Critical': 4}
incidents['severity_score'] = incidents['severity'].map(severity_map).fillna(1)

incident_features = incidents.groupby('resident_id').agg(
    incident_count=('incident_id', 'count'),
    incident_severity_score=('severity_score', 'sum'),
    unresolved_incidents=('resolved', lambda x: (x.astype(str).str.lower().isin(['no', 'false', '0'])).sum())
).reset_index()

print(f'Incident features: {incident_features.shape}')
incident_features.head()

Incident features: (44, 4)


,resident_id,incident_count,incident_severity_score,unresolved_incidents
0,1,4,7,1
1,3,2,4,2
2,4,3,5,0
3,5,2,5,1
4,6,2,3,0


### 1.7 Merge All Features into Analytical Dataset

Now we join all feature tables to the residents table on `resident_id`. Residents with no records in a particular table will have missing values, which we fill with appropriate defaults (0 for counts, median for scores).

In [11]:
# Create target variable
residents['reintegration_completed'] = (residents['reintegration_status'] == 'Completed').astype(int)

# Convert length_of_stay from string ("X Years Y months") to total months (numeric)
import re
def parse_length_of_stay(val):
    if pd.isna(val):
        return np.nan
    match = re.match(r'(\d+)\s*Years?\s+(\d+)\s*months?', str(val))
    if match:
        return int(match.group(1)) * 12 + int(match.group(2))
    return np.nan

residents['length_of_stay'] = residents['length_of_stay'].apply(parse_length_of_stay)
print(f'length_of_stay converted to numeric (months). Sample: {residents["length_of_stay"].head(5).tolist()}')

# Select resident-level features
resident_base = residents[['resident_id', 'length_of_stay', 'initial_risk_level', 
                           'current_risk_level', 'case_category', 'reintegration_completed']].copy()

# Encode risk levels
risk_map = {'Low': 1, 'Medium': 2, 'High': 3, 'Critical': 4}
resident_base['initial_risk_encoded'] = resident_base['initial_risk_level'].map(risk_map).fillna(2)
resident_base['current_risk_encoded'] = resident_base['current_risk_level'].map(risk_map).fillna(2)
resident_base['risk_improvement'] = resident_base['initial_risk_encoded'] - resident_base['current_risk_encoded']

# Merge all features
df = resident_base.merge(counseling_features, on='resident_id', how='left')
df = df.merge(edu_features, on='resident_id', how='left')
df = df.merge(health_features, on='resident_id', how='left')
df = df.merge(intervention_features, on='resident_id', how='left')
df = df.merge(visit_features, on='resident_id', how='left')
df = df.merge(incident_features, on='resident_id', how='left')

print(f'Merged analytical dataset: {df.shape}')
print(f'\nTarget distribution:')
print(df['reintegration_completed'].value_counts())
print(f'\nCompletion rate: {df["reintegration_completed"].mean():.1%}')

length_of_stay converted to numeric (months). Sample: [28, 21, 21, 17, 9]
Merged analytical dataset: (60, 31)

Target distribution:
reintegration_completed
0    41
1    19
Name: count, dtype: int64

Completion rate: 31.7%


In [12]:
df.head()

,resident_id,length_of_stay,initial_risk_level,current_risk_level,case_category,reintegration_completed,initial_risk_encoded,current_risk_encoded,risk_improvement,total_counseling_sessions,...,total_interventions,intervention_completion_rate,intervention_categories,home_visit_count,safety_concerns_count,follow_up_needed_count,home_visit_cooperation_mode,incident_count,incident_severity_score,unresolved_incidents
0,1,28,Critical,High,Neglected,0,4,3,1,106,...,3,0.0,3,54.0,9.0,27.0,Cooperative,4.0,7.0,1.0
1,2,21,Medium,Medium,Surrendered,1,2,2,0,51,...,3,0.0,3,35.0,11.0,17.0,Cooperative,NaN,NaN,NaN
2,3,21,Medium,Medium,Surrendered,1,2,2,0,53,...,3,0.0,3,26.0,11.0,12.0,Cooperative,2.0,4.0,2.0
3,4,17,High,Low,Neglected,0,3,1,2,57,...,3,0.0,3,9.0,3.0,3.0,Highly Cooperative,3.0,5.0,0.0
4,5,9,Medium,Low,Surrendered,1,2,1,1,18,...,3,0.0,3,11.0,2.0,3.0,Highly Cooperative,2.0,5.0,1.0


In [13]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 31 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   resident_id                   60 non-null     int64  
 1   length_of_stay                60 non-null     int64  
 2   initial_risk_level            60 non-null     str    
 3   current_risk_level            60 non-null     str    
 4   case_category                 60 non-null     str    
 5   reintegration_completed       60 non-null     int64  
 6   initial_risk_encoded          60 non-null     int64  
 7   current_risk_encoded          60 non-null     int64  
 8   risk_improvement              60 non-null     int64  
 9   total_counseling_sessions     60 non-null     int64  
 10  mean_session_duration         60 non-null     float64
 11  emotional_improvement_ratio   60 non-null     float64
 12  mean_education_progress       60 non-null     float64
 13  mean_attendance_ra

---
## Section 2: Univariate Profiling (Ch. 6)

With our multi-table features assembled, we profile each to understand distributions, detect data quality issues, and identify preparation needs. This is especially important here because our features were *engineered* from aggregations -- we need to verify they make sense and have sufficient variation to be useful predictors.

In [14]:
# Full univariate profile
profile = unistats(df)
profile

,dtype,count,missing_pct,unique,mean,median,std,skew,kurtosis,min,max,mode,top_freq
column,,,,,,,,,,,,,
resident_id,int64,60,0.00,60,30.5000,30.5000,17.4642,0.0000,-1.2000,1.000000,60.000000,NaN,NaN
length_of_stay,int64,60,0.00,27,18.5167,18.0000,7.9820,0.4749,-0.5795,6.000000,37.000000,NaN,NaN
initial_risk_level,str,60,0.00,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Medium,24.0
current_risk_level,str,60,0.00,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Low,34.0
case_category,str,60,0.00,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Surrendered,21.0
reintegration_completed,int64,60,0.00,2,0.3167,0.0000,0.4691,0.8086,-1.3938,0.000000,1.000000,NaN,NaN
initial_risk_encoded,int64,60,0.00,4,2.2167,2.0000,0.9037,0.2630,-0.6914,1.000000,4.000000,NaN,NaN
current_risk_encoded,int64,60,0.00,4,1.5500,1.0000,0.7231,1.2110,1.1007,1.000000,4.000000,NaN,NaN
risk_improvement,int64,60,0.00,4,0.6667,0.0000,0.8370,1.0632,0.3246,0.000000,3.000000,NaN,NaN


In [15]:
# Visualize numeric feature distributions
numeric_features = ['length_of_stay', 'total_counseling_sessions', 'emotional_improvement_ratio',
                    'mean_education_progress', 'education_trend_slope', 'mean_health_score',
                    'health_trend_slope', 'intervention_completion_rate', 'incident_count',
                    'incident_severity_score', 'home_visit_count', 'risk_improvement']

fig, axes = plt.subplots(3, 4, figsize=(18, 12))
for i, col in enumerate(numeric_features):
    if col in df.columns:
        ax = axes[i // 4, i % 4]
        df[col].hist(bins=20, ax=ax, color='#A2C9E1', edgecolor='white')
        ax.set_title(f'{col}\nskew={df[col].skew():.2f}', fontsize=9)
plt.suptitle('Feature Distributions (Ch. 6)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Section 3: Data Preparation & Cleaning (Ch. 7)

The multi-table join produces missing values for residents without records in certain tables (e.g., a resident with no incident reports has missing incident_count). We handle these systematically:
- **Count features** (incident_count, home_visit_count): Fill with 0 (no records = no incidents/visits).
- **Score/rate features** (mean_education_progress, intervention_completion_rate): Fill with median (absence of data is not the same as a score of 0).
- **Trend features** (education_trend_slope, health_trend_slope): Fill with 0 (no trend data = flat trajectory assumption).

In [16]:
# Missing value report
missing_report = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
missing_info = pd.DataFrame({'count': missing_report, 'pct': missing_pct})
print('Missing Values:')
missing_info[missing_info['count'] > 0]

Missing Values:


,count,pct
home_visit_count,2,3.33
safety_concerns_count,2,3.33
follow_up_needed_count,2,3.33
home_visit_cooperation_mode,2,3.33
incident_count,16,26.67
incident_severity_score,16,26.67
unresolved_incidents,16,26.67


In [17]:
# Fill missing values with domain-appropriate strategies (Ch. 7)
# Count features: 0 (no records = no events)
count_cols = ['total_counseling_sessions', 'education_record_count', 'health_record_count',
              'total_interventions', 'home_visit_count', 'incident_count', 
              'incident_severity_score', 'unresolved_incidents', 'safety_concerns_count',
              'follow_up_needed_count', 'intervention_categories', 'mean_session_duration']
for col in count_cols:
    if col in df.columns and df[col].isnull().any():
        df[col] = df[col].fillna(0)
        print(f'  {col}: filled with 0')

# Rate/score features: median
rate_cols = ['emotional_improvement_ratio', 'mean_education_progress', 'mean_attendance_rate',
             'mean_health_score', 'mean_nutrition_score', 'mean_sleep_quality',
             'intervention_completion_rate']
for col in rate_cols:
    if col in df.columns and df[col].isnull().any():
        df = missing_fill(df, col, strategy='median')
        print(f'  {col}: filled with median')

# Trend features: 0 (no trend = flat)
trend_cols = ['education_trend_slope', 'health_trend_slope']
for col in trend_cols:
    if col in df.columns and df[col].isnull().any():
        df[col] = df[col].fillna(0)
        print(f'  {col}: filled with 0')

# Categorical: mode
for col in ['home_visit_cooperation_mode', 'case_category']:
    if col in df.columns and df[col].isnull().any():
        df = missing_fill(df, col, strategy='mode')
        print(f'  {col}: filled with mode')

print(f'\nRemaining missing: {df.isnull().sum().sum()}')

  home_visit_count: filled with 0
  incident_count: filled with 0
  incident_severity_score: filled with 0
  unresolved_incidents: filled with 0
  safety_concerns_count: filled with 0
  follow_up_needed_count: filled with 0
  home_visit_cooperation_mode: filled with mode

Remaining missing: 0


In [18]:
# Skewness correction (Ch. 7)
skew_candidates = ['total_counseling_sessions', 'incident_count', 'incident_severity_score', 'length_of_stay']
for col in skew_candidates:
    if col in df.columns and abs(df[col].skew()) > 1:
        original_skew = df[col].skew()
        df = skew_correct(df, col, method='log')
        print(f'{col}: skew {original_skew:.2f} -> {df[col].skew():.2f}')

In [19]:
# Outlier treatment (Ch. 7)
outlier_candidates = ['mean_education_progress', 'mean_health_score']
for col in outlier_candidates:
    if col in df.columns:
        df = clean_outlier(df, col, method='iqr', factor=1.5)
        print(f'{col}: outliers capped')

mean_education_progress: outliers capped
mean_health_score: outliers capped


---
## Section 4: Bivariate Analysis (Ch. 8)

Chapter 8 analysis is especially valuable here because it reveals which dimensions of a resident's progress are most associated with successful reintegration. This directly informs caseworkers about which areas to prioritize in case management.

We use N2C analysis (numeric feature vs. binary target) for our engineered features and C2C analysis for categorical features.

In [20]:
# N2C Analysis: Each numeric feature vs reintegration_completed (Ch. 8)
n2c_features = [c for c in numeric_features if c in df.columns]

n2c_results = []
for feat in n2c_features:
    result = n2c_analysis(df, feat, 'reintegration_completed')
    result['feature'] = feat
    n2c_results.append(result)

n2c_df = pd.DataFrame(n2c_results).set_index('feature')
n2c_df['significant'] = n2c_df['p_value'] < 0.05
print('\nN2C Summary:')
n2c_df.sort_values('p_value')


N2C Summary:


,f_statistic,p_value,significant
feature,,,
home_visit_count,6.663498,0.012391,True
emotional_improvement_ratio,4.764291,0.033118,True
mean_health_score,3.254754,0.076410,False
health_trend_slope,2.703822,0.105518,False
total_counseling_sessions,1.118970,0.294526,False
education_trend_slope,0.391271,0.534086,False
length_of_stay,0.207287,0.650602,False
incident_count,0.017524,0.895143,False
risk_improvement,0.012011,0.913108,False


**N2C Interpretation (Ch. 8):** The ANOVA results identify which progress dimensions show statistically different distributions between successfully reintegrated residents and those still in care. Features with the lowest p-values are the strongest differentiators.

We expect intervention_completion_rate and risk_improvement to be among the strongest signals, as they directly measure treatment progress and risk reduction. Education and health trend slopes may also be significant, as positive trajectories indicate readiness for independent living.

In [21]:
# C2C Analysis: Categorical features vs reintegration_completed (Ch. 8)
cat_features = ['home_visit_cooperation_mode', 'case_category']
for feat in cat_features:
    if feat in df.columns:
        result = c2c_analysis(df, feat, 'reintegration_completed')
        print(f'{feat} vs reintegration_completed: chi2={result["chi2"]:.2f}, p={result["p_value"]:.4f}\n')

home_visit_cooperation_mode vs reintegration_completed: chi2=0.06, p=0.8065

case_category vs reintegration_completed: chi2=5.71, p=0.1265



In [22]:
# Correlation matrix (Ch. 8)
corr_cols = [c for c in numeric_features if c in df.columns]
corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax, annot_kws={'size': 8})
ax.set_title('Feature Correlation Matrix (Ch. 8)', fontsize=13)
plt.tight_layout()
plt.show()

---
## Section 5: Model Building (Ch. 9, 10-12)

### Phase 1: Explanatory Logistic Regression (Ch. 9)

The explanatory model helps caseworkers understand which factors contribute most to successful reintegration. The odds ratios translate complex statistical relationships into intuitive statements like: "Completing 80% of intervention plans increases the odds of successful reintegration by X times."

### Phase 2: Predictive Classification (Ch. 10-12)

We train Decision Tree, Logistic Regression, and Gradient Boosting classifiers, all evaluated with StratifiedKFold cross-validation. The key difference from Pipeline 1 is our emphasis on **precision** -- we need to minimize false positives (incorrectly predicting readiness).

In [23]:
# Prepare modeling dataset
feature_cols = [c for c in numeric_features if c in df.columns]
cat_model_cols = [c for c in cat_features if c in df.columns]

df_model = df[feature_cols + cat_model_cols + ['reintegration_completed']].copy()
df_model = pd.get_dummies(df_model, columns=cat_model_cols, drop_first=True, dtype=int)

X = df_model.drop(columns=['reintegration_completed'])
y = df_model['reintegration_completed']

print(f'X shape: {X.shape}')
print(f'Features: {list(X.columns)}')
print(f'y distribution: {y.value_counts().to_dict()}')

X shape: (60, 16)
Features: ['length_of_stay', 'total_counseling_sessions', 'emotional_improvement_ratio', 'mean_education_progress', 'education_trend_slope', 'mean_health_score', 'health_trend_slope', 'intervention_completion_rate', 'incident_count', 'incident_severity_score', 'home_visit_count', 'risk_improvement', 'home_visit_cooperation_mode_Highly Cooperative', 'case_category_Foundling', 'case_category_Neglected', 'case_category_Surrendered']
y distribution: {0: 41, 1: 19}


In [24]:
import statsmodels.api as sm

# Explanatory Logistic Regression (Ch. 9)
X_sm = sm.add_constant(X.astype(float))
logit_model = sm.Logit(y, X_sm)
logit_result = logit_model.fit(method='bfgs', maxiter=200, disp=0)

print(logit_result.summary2())

                                                 Results: Logit
Model:                             Logit                                Pseudo R-squared:              0.453    
Dependent Variable:                reintegration_completed              AIC:                           72.9458  
Date:                              2026-04-06 17:30                     BIC:                           106.4553 
No. Observations:                  60                                   Log-Likelihood:                -20.473  
Df Model:                          15                                   LL-Null:                       -37.460  
Df Residuals:                      44                                   LLR p-value:                   0.0034342
Converged:                         1.0000                               Scale:                         1.0000   
Method:                            MLE                                                                          
--------------------------------

In [25]:
# Odds ratios
odds_ratios = pd.DataFrame({
    'coef': logit_result.params,
    'odds_ratio': np.exp(logit_result.params),
    'p_value': logit_result.pvalues,
    'ci_lower': np.exp(logit_result.conf_int()[0]),
    'ci_upper': np.exp(logit_result.conf_int()[1])
})
odds_ratios['significant'] = odds_ratios['p_value'] < 0.05
print('Odds Ratios for Reintegration Readiness (Ch. 9):')
odds_ratios.sort_values('p_value')

Odds Ratios for Reintegration Readiness (Ch. 9):


,coef,odds_ratio,p_value,ci_lower,ci_upper,significant
home_visit_count,0.174977,1.191218e+00,0.010719,1.041414e+00,1.362572e+00,True
length_of_stay,-0.544970,5.798593e-01,0.031526,3.528595e-01,9.528914e-01,True
emotional_improvement_ratio,15.687359,6.500301e+06,0.033675,3.357428e+00,1.258520e+13,True
total_counseling_sessions,0.122652,1.130491e+00,0.067354,9.912776e-01,1.289255e+00,False
incident_count,-2.093330,1.232759e-01,0.102998,9.955218e-03,1.526531e+00,False
case_category_Surrendered,2.204918,9.069508e+00,0.115965,5.802609e-01,1.417569e+02,False
home_visit_cooperation_mode_Highly Cooperative,2.676731,1.453750e+01,0.127686,4.642556e-01,4.552208e+02,False
mean_health_score,-5.407443,4.483087e-03,0.232290,6.282993e-07,3.198805e+01,False
case_category_Neglected,-1.768672,1.705593e-01,0.241158,8.859925e-03,3.283378e+00,False
const,9.912687,2.018484e+04,0.444053,1.908270e-07,2.135062e+15,False


**Explanatory Model Interpretation (Ch. 9):**

The odds ratios reveal which dimensions of a resident's care journey are most associated with successful reintegration:

- **intervention_completion_rate:** If significant with OR > 1, this confirms that completing prescribed treatment plans is the single most important predictor of readiness. This validates the case management framework and emphasizes the importance of plan adherence.
- **risk_improvement:** A positive coefficient means residents whose risk level decreased during care have higher odds of successful reintegration. This validates the risk assessment process.
- **emotional_improvement_ratio:** Therapeutic responsiveness (improving emotional states in counseling sessions) may independently predict readiness beyond simple session counts.
- **incident_count/severity:** Negative coefficients indicate that behavioral incidents reduce the odds of successful reintegration, emphasizing the importance of behavioral stability.
- **education/health trends:** Positive slopes (improving trajectories) may be more predictive than absolute scores, as they capture momentum toward readiness.

In [26]:
# Visualize significant odds ratios
sig_or = odds_ratios[(odds_ratios['significant']) & (odds_ratios.index != 'const')].sort_values('odds_ratio')
if len(sig_or) > 0:
    fig, ax = plt.subplots(figsize=(10, max(4, len(sig_or) * 0.5)))
    colors = ['#27AE60' if x > 1 else '#E74C3C' for x in sig_or['odds_ratio']]
    ax.barh(sig_or.index, sig_or['odds_ratio'], color=colors)
    ax.axvline(x=1, color='black', linestyle='--')
    ax.set_xlabel('Odds Ratio')
    ax.set_title('Significant Predictors of Reintegration Readiness (Ch. 9)\nGreen = increases odds, Red = decreases odds')
    plt.tight_layout()
    plt.show()
else:
    print('No individually significant coefficients at p < 0.05.')

In [27]:
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, 
                             roc_auc_score, classification_report, confusion_matrix, 
                             RocCurveDisplay)

# Stratified K-Fold
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Models
models = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(random_state=42, max_iter=1000))
    ]),
    'Decision Tree': DecisionTreeClassifier(random_state=42, max_depth=5),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42, n_estimators=100, max_depth=3)
}

# Cross-validate
scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
results = {}

for name, model in models.items():
    cv_results = cross_validate(model, X, y, cv=cv, scoring=scoring, return_train_score=True)
    results[name] = {
        'accuracy': cv_results['test_accuracy'].mean(),
        'precision': cv_results['test_precision'].mean(),
        'recall': cv_results['test_recall'].mean(),
        'f1': cv_results['test_f1'].mean(),
        'auc_roc': cv_results['test_roc_auc'].mean(),
        'train_accuracy': cv_results['train_accuracy'].mean()
    }
    print(f'{name:25s} | Acc: {results[name]["accuracy"]:.3f} | PREC: {results[name]["precision"]:.3f} | '
          f'Rec: {results[name]["recall"]:.3f} | F1: {results[name]["f1"]:.3f} | AUC: {results[name]["auc_roc"]:.3f}')

results_df = pd.DataFrame(results).T
print('\nNote: PRECISION is the priority metric for this use case (minimize false positives).')
results_df

Logistic Regression       | Acc: 0.683 | PREC: 0.500 | Rec: 0.600 | F1: 0.541 | AUC: 0.716
Decision Tree             | Acc: 0.683 | PREC: 0.406 | Rec: 0.517 | F1: 0.442 | AUC: 0.628


Gradient Boosting         | Acc: 0.733 | PREC: 0.587 | Rec: 0.617 | F1: 0.571 | AUC: 0.825

Note: PRECISION is the priority metric for this use case (minimize false positives).


,accuracy,precision,recall,f1,auc_roc,train_accuracy
Logistic Regression,0.683333,0.500000,0.600000,0.541429,0.716435,0.825000
Decision Tree,0.683333,0.405714,0.516667,0.442424,0.628472,0.991667
Gradient Boosting,0.733333,0.586667,0.616667,0.571429,0.825463,1.000000


In [28]:
# Visual comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

results_df[['accuracy', 'precision', 'recall', 'f1', 'auc_roc']].plot(kind='bar', ax=axes[0])
axes[0].set_title('Model Comparison - Reintegration Readiness (Ch. 10-12)\nPrecision is the priority')
axes[0].set_ylabel('Score')
axes[0].set_ylim(0, 1.05)
axes[0].legend(loc='lower right', fontsize=8)
plt.sca(axes[0])
plt.xticks(rotation=45, ha='right')

# Highlight precision
axes[1].bar(results_df.index, results_df['precision'], color='#E74C3C', alpha=0.7, label='Precision')
axes[1].bar(results_df.index, results_df['recall'], color='#3498DB', alpha=0.5, label='Recall')
axes[1].set_title('Precision vs Recall Tradeoff\n(Precision = minimize premature reintegration)')
axes[1].set_ylabel('Score')
axes[1].legend()
plt.sca(axes[1])
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.show()

In [29]:
# ROC Curves
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

fig, ax = plt.subplots(figsize=(8, 6))
for name, model in models.items():
    model.fit(X_train, y_train)
    RocCurveDisplay.from_estimator(model, X_test, y_test, name=name, ax=ax)

ax.plot([0, 1], [0, 1], 'k--', label='Random (AUC=0.5)')
ax.set_title('ROC Curves - Reintegration Models (Ch. 10-12)')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

---
## Section 6: Model Tuning, Selection & Deployment (Ch. 13-15)

We tune the Gradient Boosting model with GridSearchCV, **optimizing for precision** rather than the default accuracy. This reflects our business priority: we must minimize the risk of premature reintegration (false positives).

A model with 95% precision and 60% recall is preferable to one with 85% precision and 90% recall for this specific use case, because the cost of a false positive (child returned to an unsafe environment) far exceeds the cost of a false negative (delaying reintegration for a ready child).

In [30]:
from sklearn.model_selection import GridSearchCV

# GridSearchCV optimizing for PRECISION (Ch. 13)
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [2, 3, 5],
    'learning_rate': [0.05, 0.1, 0.2],
    'min_samples_split': [2, 5, 10]
}

gb = GradientBoostingClassifier(random_state=42)
grid_search = GridSearchCV(gb, param_grid, cv=cv, scoring='precision', n_jobs=-1, verbose=0)
grid_search.fit(X_train, y_train)

print(f'Best Parameters (optimized for precision): {grid_search.best_params_}')
print(f'Best CV Precision: {grid_search.best_score_:.4f}')

Best Parameters (optimized for precision): {'learning_rate': 0.1, 'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 50}
Best CV Precision: 0.5933


In [31]:
# Evaluate best model on test set
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

print('Best Model - Test Set Performance:')
print(f'  Accuracy:  {accuracy_score(y_test, y_pred):.4f}')
print(f'  PRECISION: {precision_score(y_test, y_pred, zero_division=0):.4f}  <-- priority metric')
print(f'  Recall:    {recall_score(y_test, y_pred, zero_division=0):.4f}')
print(f'  F1 Score:  {f1_score(y_test, y_pred, zero_division=0):.4f}')
print(f'  AUC-ROC:   {roc_auc_score(y_test, y_proba):.4f}')
print(f'\n{classification_report(y_test, y_pred, zero_division=0)}')

Best Model - Test Set Performance:
  Accuracy:  0.6667
  PRECISION: 0.5000  <-- priority metric
  Recall:    0.5000
  F1 Score:  0.5000
  AUC-ROC:   0.6875

              precision    recall  f1-score   support

           0       0.75      0.75      0.75         8
           1       0.50      0.50      0.50         4

    accuracy                           0.67        12
   macro avg       0.62      0.62      0.62        12
weighted avg       0.67      0.67      0.67        12



In [32]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Not Ready', 'Ready'], yticklabels=['Not Ready', 'Ready'])
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix - Reintegration Model\n(Minimize top-right: false positives)')
plt.tight_layout()
plt.show()

print(f'False Positives (predicted ready but not ready): {cm[0, 1]}')
print(f'False Negatives (predicted not ready but actually ready): {cm[1, 0]}')

False Positives (predicted ready but not ready): 2
False Negatives (predicted not ready but actually ready): 2


In [33]:
# Permutation importance (Ch. 14)
from sklearn.inspection import permutation_importance

perm_imp = permutation_importance(best_model, X_test, y_test, n_repeats=10, random_state=42, scoring='precision')
perm_df = pd.DataFrame({
    'feature': X.columns,
    'importance_mean': perm_imp.importances_mean,
    'importance_std': perm_imp.importances_std
}).sort_values('importance_mean', ascending=False)

print('Permutation Feature Importance (optimized for precision, Ch. 14):')
perm_df

Permutation Feature Importance (optimized for precision, Ch. 14):


,feature,importance_mean,importance_std
10,home_visit_count,0.266667,0.334996
2,emotional_improvement_ratio,0.175000,0.025000
5,mean_health_score,0.020000,0.040000
1,total_counseling_sessions,0.000000,0.000000
7,intervention_completion_rate,0.000000,0.000000
4,education_trend_slope,0.000000,0.000000
8,incident_count,0.000000,0.000000
0,length_of_stay,0.000000,0.063246
12,home_visit_cooperation_mode_Highly Cooperative,0.000000,0.000000
13,case_category_Foundling,0.000000,0.000000


In [34]:
# Feature importance visualization
fig, ax = plt.subplots(figsize=(10, max(4, len(perm_df) * 0.35)))
ax.barh(perm_df['feature'], perm_df['importance_mean'],
        xerr=perm_df['importance_std'], color='#91B191')
ax.set_xlabel('Mean Permutation Importance (Precision)')
ax.set_title('Feature Importance - Reintegration Model (Ch. 14)')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [35]:
# Deploy: Save model (Ch. 15)
import joblib

# Retrain on full dataset
final_model = GradientBoostingClassifier(random_state=42, **grid_search.best_params_)
final_model.fit(X, y)

model_path = 'models/reintegration_model.pkl'
joblib.dump(final_model, model_path)
print(f'Model saved to {model_path}')
print(f'Model type: {type(final_model).__name__}')
print(f'Parameters: {final_model.get_params()}')

# Verify loading
loaded = joblib.load(model_path)
test_pred = loaded.predict(X.head(5))
print(f'\nVerification - predictions on first 5: {test_pred}')
print('Deployment successful.')

Model saved to models/reintegration_model.pkl
Model type: GradientBoostingClassifier
Parameters: {'ccp_alpha': 0.0, 'criterion': 'friedman_mse', 'init': None, 'learning_rate': 0.1, 'loss': 'log_loss', 'max_depth': 5, 'max_features': None, 'max_leaf_nodes': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'n_estimators': 50, 'n_iter_no_change': None, 'random_state': 42, 'subsample': 1.0, 'tol': 0.0001, 'validation_fraction': 0.1, 'verbose': 0, 'warm_start': False}

Verification - predictions on first 5: [0 1 1 0 1]
Deployment successful.


---
## Summary & Business Recommendations

### Key Findings

This pipeline built a reintegration readiness prediction model by aggregating features from seven interconnected tables spanning counseling, education, health, intervention plans, home visitations, and incident reports. The model prioritizes **precision** to minimize the risk of premature reintegration.

### Business Recommendations for Social Workers

1. **Decision Support Dashboard:** Deploy the model as a scoring tool in the case management system. Each resident receives a readiness probability score that social workers can review alongside their professional assessment. The model should NEVER be the sole decision-maker -- it is a supplement to clinical judgment.

2. **Multi-Dimensional Progress Tracking:** The feature importance analysis reveals which dimensions matter most for readiness. Social workers should ensure comprehensive data collection across all dimensions -- incomplete data reduces model accuracy.

3. **Trend Monitoring:** If education and health trend slopes are important features, this validates the value of longitudinal tracking. A resident with improving trajectories may be more ready than one with higher but static scores.

4. **Incident Response Protocol:** If incident count/severity are strong negative predictors, the organization should invest in behavioral support programs and ensure incidents are thoroughly documented for accurate modeling.

5. **Family Assessment Weight:** If home visitation features are significant, this underscores that readiness depends not just on the resident's progress but on the family environment's capacity to provide safety and support.

### Ethical Considerations

- The model encodes historical patterns that may reflect systemic biases in how different case categories are handled.
- The high-stakes nature of reintegration decisions means the model should always be paired with human oversight.
- Regular model audits should check for disparate impact across demographic groups.